# Evaluación Rápida de Modelos SegRot

Este notebook proporciona una evaluación rápida de los modelos con visualizaciones.
Incluye:
- Métricas esenciales (IoU, Accuracy)
- Visualizaciones de predicciones
- Comparación rápida entre modelos

In [ ]:
# Imports y configuración
from model import MobilnetSegRot, ResNet50SegRot, EfficientNetSegRot
import torch
import torch.nn as nn
import pandas as pd
from pathlib import Path
from utils import Load_Dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# Configuración
torch.manual_seed(17)
plt.style.use('default')

print("📚 Librerías cargadas")
print(f"🖥️ Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# Función para calcular métricas en batch
def calculate_metrics_batch(pred_mask, true_mask, pred_class, true_class):
    """Calcula métricas para un batch completo"""
    # Métricas de segmentación
    pred_mask_sigmoid = torch.sigmoid(pred_mask)
    pred_mask_binary = (pred_mask_sigmoid > 0.5).float()
    
    # IoU por imagen
    ious = []
    for i in range(pred_mask.size(0)):
        pred_flat = pred_mask_binary[i].view(-1).bool()
        true_flat = true_mask[i].view(-1).bool()
        
        intersection = (pred_flat & true_flat).float().sum()
        union = (pred_flat | true_flat).float().sum()
        
        iou = (intersection + 1e-6) / (union + 1e-6)
        ious.append(iou.item())
    
    # Accuracy de clasificación
    pred_classes = pred_class.argmax(dim=1)
    correct_predictions = (pred_classes == true_class).float()
    accuracy = correct_predictions.mean().item()
    
    return ious, accuracy, pred_classes.cpu().numpy(), true_class.cpu().numpy()

print("✅ Función de métricas definida")

In [ ]:
# Cargar dataset
df_path = Path("../Data_df/progreso_orientacion_aument.csv").resolve()
path = Path("../Data_img/imagenes_cedula_peq").resolve()
results_dir = Path("test_results")
results_dir.mkdir(exist_ok=True)

print(f"📁 Dataset: {df_path}")
print(f"📁 Imágenes: {path}")

# Cargar datos
df = pd.read_csv(df_path).drop(columns=["Unnamed: 0"], errors="ignore")
uniques = np.unique(df['Clase'].values)
num_classes = len(uniques)

print(f"📊 Dataset: {len(df)} muestras, {num_classes} clases")

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")

# Dataset
dataset = Load_Dataset(df, path, batch_size=8, num_workers=2)
test_dataloader = dataset.load_test()
print(f"📦 Test batches: {len(test_dataloader)}")

In [ ]:
# Función de evaluación rápida
def quick_evaluate_model(model, model_name, test_dataloader, device):
    """Evaluación rápida de un modelo"""
    print(f"\n⚡ Evaluación rápida: {model_name}")
    
    model.eval()
    all_ious = []
    all_accuracies = []
    total_samples = 0
    
    with torch.no_grad():
        for img, mask, clase in tqdm(test_dataloader, desc=f"Testing {model_name}"):
            img = img.to(device).float()
            mask = mask.to(device).float()
            clase = clase.to(device).long()
            
            pred_mask, pred_class = model(img)
            
            ious, accuracy, _, _ = calculate_metrics_batch(pred_mask, mask, pred_class, clase)
            
            all_ious.extend(ious)
            all_accuracies.append(accuracy)
            total_samples += img.size(0)
    
    mean_iou = np.mean(all_ious)
    mean_accuracy = np.mean(all_accuracies)
    
    return {
        'model': model_name,
        'samples': total_samples,
        'mean_iou': mean_iou,
        'std_iou': np.std(all_ious),
        'accuracy': mean_accuracy,
        'min_iou': np.min(all_ious),
        'max_iou': np.max(all_ious)
    }

print("✅ Función de evaluación rápida definida")

In [ ]:
# Función de visualización
def visualize_predictions(model, model_name, test_dataloader, device, num_samples=6):
    """Visualiza predicciones del modelo"""
    model.eval()
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    sample_count = 0
    
    with torch.no_grad():
        for img, mask, clase in test_dataloader:
            if sample_count >= num_samples:
                break
                
            img = img.to(device).float()
            mask = mask.to(device).float()
            clase = clase.to(device).long()
            
            pred_mask, pred_class = model(img)
            pred_mask_sigmoid = torch.sigmoid(pred_mask)
            pred_mask_binary = (pred_mask_sigmoid > 0.5).float()
            
            # Tomar solo las primeras imágenes del batch
            batch_size = min(img.size(0), num_samples - sample_count)
            
            for i in range(batch_size):
                row = sample_count + i
                
                # Imagen original
                img_np = img[i].cpu().permute(1, 2, 0).numpy()
                img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min())
                axes[row, 0].imshow(img_np)
                axes[row, 0].set_title('Original')
                axes[row, 0].axis('off')
                
                # Máscara verdadera
                axes[row, 1].imshow(mask[i].cpu().squeeze(), cmap='gray')
                axes[row, 1].set_title('Máscara Real')
                axes[row, 1].axis('off')
                
                # Máscara predicha (probabilidades)
                axes[row, 2].imshow(pred_mask_sigmoid[i].cpu().squeeze(), cmap='hot', vmin=0, vmax=1)
                axes[row, 2].set_title('Predicción (Prob.)')
                axes[row, 2].axis('off')
                
                # Máscara predicha (binaria) + IoU
                axes[row, 3].imshow(pred_mask_binary[i].cpu().squeeze(), cmap='gray')
                
                # Calcular IoU para esta muestra
                ious, _, _, _ = calculate_metrics_batch(
                    pred_mask[i:i+1], mask[i:i+1], 
                    pred_class[i:i+1], clase[i:i+1]
                )
                iou = ious[0]
                
                # Agregar información al título
                pred_class_idx = pred_class[i].argmax().item()
                true_class_idx = clase[i].item()
                axes[row, 3].set_title(f'Pred. (IoU: {iou:.3f})\nClase: {pred_class_idx} (Real: {true_class_idx})')
                axes[row, 3].axis('off')
            
            sample_count += batch_size
            if sample_count >= num_samples:
                break
    
    plt.suptitle(f'Predicciones de {model_name}', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Guardar figura
    save_path = results_dir / f'predictions_{model_name}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 Visualizaciones guardadas: {save_path}")
    plt.show()

print("✅ Función de visualización definida")

In [ ]:
# Definir modelos a evaluar
models_to_test = [
    ("MobilnetSegRot", MobilnetSegRot(num_classes=num_classes), "MobilnetSegRot.pth"),
    ("ResNet50SegRot", ResNet50SegRot(num_classes=num_classes), "ResNet50SegRot.pth"),
    ("EfficientNetSegRot", EfficientNetSegRot(num_classes=num_classes), "EfficientNetSegRot.pth")
]

print("🤖 Verificando modelos disponibles:")
for model_name, _, model_path in models_to_test:
    exists = "✅" if os.path.isfile(model_path) else "❌"
    print(f"   {exists} {model_name}: {model_path}")

In [ ]:
# EVALUACIÓN RÁPIDA DE TODOS LOS MODELOS
results = []

for model_name, model, model_path in models_to_test:
    if os.path.isfile(model_path):
        print(f"\n🔄 Cargando {model_name}...")
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        
        # Evaluación rápida
        result = quick_evaluate_model(model, model_name, test_dataloader, device)
        results.append(result)
        
        print(f"✅ {model_name}:")
        print(f"   IoU: {result['mean_iou']:.4f} ± {result['std_iou']:.4f}")
        print(f"   Accuracy: {result['accuracy']:.4f}")
        print(f"   Rango IoU: [{result['min_iou']:.4f}, {result['max_iou']:.4f}]")
        
        # Limpiar memoria
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    else:
        print(f"❌ No se encontró {model_path}")

In [ ]:
# VISUALIZACIÓN DE PREDICCIONES
print("\n🎨 GENERANDO VISUALIZACIONES...")

for model_name, model, model_path in models_to_test:
    if os.path.isfile(model_path):
        print(f"\n📊 Visualizando {model_name}...")
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        
        # Generar visualizaciones
        visualize_predictions(model, model_name, test_dataloader, device, num_samples=4)
        
        # Limpiar memoria
        del model
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    else:
        print(f"❌ Saltando {model_name} (archivo no encontrado)")

In [ ]:
# GUARDAR RESULTADOS Y COMPARACIÓN FINAL
if results:
    results_df = pd.DataFrame(results)
    results_path = results_dir / "quick_evaluation_results.csv"
    results_df.to_csv(results_path, index=False)
    print(f"\n📊 Resultados guardados: {results_path}")
    
    # Mostrar tabla de resultados
    print("\n📊 TABLA DE RESULTADOS:")
    display(results_df)
    
    # Comparación final
    print(f"\n🏆 COMPARACIÓN RÁPIDA:")
    for _, row in results_df.iterrows():
        print(f"   {row['model']}: IoU={row['mean_iou']:.4f}, Acc={row['accuracy']:.4f}")
    
    # Mejores modelos
    best_iou_model = results_df.loc[results_df['mean_iou'].idxmax()]
    best_acc_model = results_df.loc[results_df['accuracy'].idxmax()]
    
    print(f"\n🥇 MEJORES MODELOS:")
    print(f"   Mejor IoU: {best_iou_model['model']} ({best_iou_model['mean_iou']:.4f})")
    print(f"   Mejor Accuracy: {best_acc_model['model']} ({best_acc_model['accuracy']:.4f})")

print(f"\n📁 Archivos guardados en: {results_dir}")
print("⚡ Evaluación rápida completada!")